In [1]:
!pip -q uninstall -y transformers tokenizers
!pip -q install tokenizer "transformers==4.57.0"  accelerate safetensors sentencepiece

Reason for being yanked: Error in the setup causing installation issues

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [20]:
%pip show transformers

Name: transformers
Version: 4.57.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /Users/gavinye/Efficient-dLLM-Inference/dlm_env/lib/python3.14/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
torch.bool

NameError: name 'torch' is not defined

In [ ]:
import transformers.modeling_rope_utils as rope_utils
import torch
# Keep references to originals
_orig_linear = rope_utils._compute_linear_scaling_rope_parameters

def _linear_with_default_factor(config, device, seq_len=None, layer_type=None, **rope_kwargs):
    # Ensure rope_scaling exists and has factor (no-op scaling)
    if config is not None and hasattr(config, "rope_scaling") and isinstance(config.rope_scaling, dict):
        # Transformers linear scaling expects config.rope_scaling["factor"]
        config.rope_scaling.setdefault("factor", 1.0)
        # Also normalize "rope_type" -> "type" if needed
        if "type" not in config.rope_scaling and "rope_type" in config.rope_scaling:
            config.rope_scaling["type"] = config.rope_scaling.pop("rope_type")
    return _orig_linear(config, device, seq_len=seq_len, layer_type=layer_type, **rope_kwargs)

# 1) Patch the module function (nice-to-have)
rope_utils._compute_linear_scaling_rope_parameters = _linear_with_default_factor

# 2) Patch the mapping used by the remote model (this is the critical part)
rope_utils.ROPE_INIT_FUNCTIONS["linear"] = _linear_with_default_factor

# 3) Optional safety: if remote code uses rope_type="default", map it too
if "default" not in rope_utils.ROPE_INIT_FUNCTIONS:
    rope_utils.ROPE_INIT_FUNCTIONS["default"] = _linear_with_default_factor

print("Patched ROPE_INIT_FUNCTIONS keys:", sorted(list(rope_utils.ROPE_INIT_FUNCTIONS.keys()))[:10], "...")

Patched ROPE_INIT_FUNCTIONS keys: ['default', 'dynamic', 'linear', 'llama3', 'longrope', 'yarn'] ...


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "inclusionAI/LLaDA2.1-mini"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, device_map="auto")

/Users/gavinye/Efficient-dLLM-Inference/dlm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 8/8 [00:35<00:00,  4.49s/it]
Some parameters are on the meta device because they were offloaded to the disk.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

In [6]:
import torch
import torch.nn.functional as F

_builtin_sdpa = torch._C._nn.scaled_dot_product_attention  # always the true builtin

def sdpa_cast_mask(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False):
    if attn_mask is not None and attn_mask.dtype is not torch.bool and attn_mask.dtype != query.dtype:
        attn_mask = attn_mask.to(dtype=query.dtype)
    return _builtin_sdpa(
        query, key, value,
        attn_mask=attn_mask,
        dropout_p=dropout_p,
        is_causal=is_causal,
        scale=scale,
        enable_gqa=enable_gqa,
    )

F.scaled_dot_product_attention = sdpa_cast_mask
print("Installed sdpa_cast_mask patch (non-recursive).")

Installed sdpa_cast_mask patch (non-recursive).


In [7]:
F.scaled_dot_product_attention

<function __main__.sdpa_cast_mask(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False)>

In [8]:
import torch

B,H,T,D = 1,1,4,8
q = torch.randn(B,H,T,D, dtype=torch.float32)
k = torch.randn(B,H,T,D, dtype=torch.float32)
v = torch.randn(B,H,T,D, dtype=torch.float32)
mask = torch.ones(B,H,T,T, dtype=torch.bfloat16)

# If patch is not installed, many torch versions error here.
out = F.scaled_dot_product_attention(q,k,v, attn_mask=mask)
print("OK:", out.shape, out.dtype)

OK: torch.Size([1, 1, 4, 8]) torch.float32


In [11]:
model.device

device(type='mps', index=0)

In [19]:
# GSM8K-style question
prompt = """A store sells notebooks for $3 each and pens for $2 each.
Mia buys 4 notebooks and 7 pens. She pays with a $30 bill.
How much change does she get? Explain your steps."""

# LLaDA2.1-mini expects chat-format prompting
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
)

# Move input to the same device as the model’s first parameter shard
# (works with device_map="auto")
input_ids = input_ids.to(model.device)

with torch.no_grad():
    generated_tokens = model.generate(
        inputs=input_ids,
        eos_early_stop=True,
        gen_length=16,
        block_length=4,
        steps=20,
        threshold=0.5,
        editing_threshold=0,
        temperature=0.0,
    )

text = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
print(text)

Let's solve the problem step by step.

### Step 1:
